#  Training a New Tokenizer from an Old One

**Source:** [HuggingFace LLM Course – Chapter 6, Section 2](https://huggingface.co/learn/llm-course/chapter6/2)

If you want to train a language model from scratch on a specific domain (like code) or a new language, you should also train a custom tokenizer. A tokenizer trained on standard English text (like the GPT-2 tokenizer) is inefficient at tokenizing Python code. 

In this notebook, we will:
1. Assemble a massive training corpus of Python code using the `code_search_net` dataset.
2. Learn how to iterate over massive datasets memory-efficiently using Python generators.
3. Use the blazing-fast Rust backend of Hugging Face Tokenizers to train a custom Python tokenizer.
4. Compare the tokenization efficiency of the new tokenizer vs. the old English tokenizer.
5. Save and push our new tokenizer to the Hugging Face Hub.

---

##  Step 1: Install Required Libraries

We need to install the following libraries to train our tokenizer and download the dataset:
- **`datasets`**: For downloading and processing the `code_search_net` dataset.
- **`transformers[sentencepiece]`**: For tokenization utilities.
- **`evaluate`**: For evaluation metrics (if needed later).

> 💡 Uncomment and run this cell if you haven't installed these libraries yet.

In [ ]:
#!pip install datasets evaluate transformers[sentencepiece]
#!apt install git-lfs

##  Step 2: Configure Git

Set up your global Git configuration. This is necessary because saving and pushing tokenizers to the Hugging Face Hub uses Git under the hood.

In [1]:
!git config --global user.email "sujatkhan24@gmail.com"
!git config --global user.name "sak-07"

##  Step 3: Login to Hugging Face Hub

Run `notebook_login()` to authenticate with the Hugging Face Hub. 
You will need an access token (with write permissions) from your Hugging Face account settings to push the trained tokenizer later.

In [2]:
from huggingface_hub import notebook_login

notebook_login()

##  Step 4: Load the Python Code Dataset

To train a tokenizer for Python code, we need a large corpus of Python code. We'll use the **`code_search_net`** dataset.

- It contains millions of functions from open-source GitHub repositories.
- We specifically load the `"python"` subset.

> ⏳ This might take a few minutes as it downloads a large amount of data.

In [3]:
from datasets import load_dataset

# This can take a few minutes to load, so grab a coffee or tea while you wait!
raw_datasets = load_dataset("code_search_net", "python")

README.md: 0.00B [00:00, ?B/s]

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sujat\.cache\huggingface\hub\datasets--code_search_net. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/28.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

##  Step 5: Inspect the Dataset

Let's look at the `train` split of the dataset. 
You can see it contains over 400,000 rows with columns like `func_name`, `whole_func_string`, `func_documentation_string`, etc.

In [4]:
raw_datasets["train"]

Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 412178
})

##  Step 6: View a Sample Function

We'll train our tokenizer using the `whole_func_string` column, which contains the raw source code of the functions.
Let's print an example function from the dataset to see what our training data looks like.

In [5]:
print(raw_datasets["train"][123456]["whole_func_string"])

def oauth_token_create(self, data, **kwargs):
        "https://developer.zendesk.com/rest_api/docs/core/oauth_tokens#create-token"
        api_path = "/api/v2/oauth/tokens.json"
        return self.call(api_path, method="POST", data=data, **kwargs)


##  Step 7: The Naive Approach (List Comprehension)

To train a tokenizer, we need to pass it batches of text. 
We *could* load everything into memory using a list comprehension like the code below, but **this is a bad idea for large datasets** because it would exhaust your RAM.

> 🛑 Leave this commented out!

In [7]:
# Don't uncomment the following line unless your dataset is small!
# training_corpus = [raw_datasets["train"][i: i + 1000]["whole_func_string"] for i in range(0, len(raw_datasets["train"]), 1000)]

##  Step 8: The Memory-Efficient Approach (Generators)

Instead of a list, we use a **Python generator expression** (indicated by parentheses `(...)` instead of square brackets `[...]`).

- A generator doesn't load everything into memory at once.
- It only loads 1,000 texts at a time when the tokenizer requests the next batch.
- This is highly efficient and allows you to process massive datasets.

In [6]:
training_corpus = (
    raw_datasets["train"][i : i + 1000]["whole_func_string"]
    for i in range(0, len(raw_datasets["train"]), 1000)
)

##  Step 9: The Problem with Generators

A generator object can only be iterated over **once**.
As demonstrated below, the first time we convert `gen` to a list, we get the numbers 0-9. The second time, we get an empty list `[]` because the generator is exhausted.

This means if we need to iterate over the training corpus multiple times, our generator expression approach will fail.

In [7]:
gen = (i for i in range(10))
print(list(gen))
print(list(gen))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


##  Step 10: Using a Generator Function

To solve the exhaustion problem, we wrap the generator expression inside a **function**. 
Every time we call `get_training_corpus()`, it returns a *fresh* generator object, allowing us to iterate over the dataset from the beginning as many times as needed.

In [8]:
def get_training_corpus():
    return (
        raw_datasets["train"][i : i + 1000]["whole_func_string"]
        for i in range(0, len(raw_datasets["train"]), 1000)
    )


training_corpus = get_training_corpus()

##  Step 11: Alternative Generator Function (Using `yield`)

We can achieve the exact same result using a standard `for` loop and the **`yield`** keyword.
This approach is often more readable and allows for complex filtering or processing logic inside the loop before yielding the batch of texts.

In [9]:
def get_training_corpus():
    dataset = raw_datasets["train"]
    for start_idx in range(0, len(dataset), 1000):
        samples = dataset[start_idx : start_idx + 1000]
        yield samples["whole_func_string"]

##  Step 12: Load an Existing Tokenizer

Even though we are training a *new* tokenizer from scratch, we use the `GPT-2` tokenizer as a starting point.

Why?
- We keep the same tokenization rules and special tokens.
- We only want to learn a **new vocabulary** that is optimized for Python code rather than standard English text.

In [10]:
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sujat\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

##  Step 13: Test the Old Tokenizer on Python Code

Let's see how the standard English `GPT-2` tokenizer handles Python code.

**Issues with the old tokenizer:**
- It represents spaces individually (the `Ġ` symbol).
- It doesn't group indentations (like 4 spaces) together.
- It splits variable names strangely (e.g., `add_numbers` becomes `Ġadd`, `_`, `n`, `umbers`).

In [11]:
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = old_tokenizer.tokenize(example)
tokens

['def',
 'Ġadd',
 '_',
 'n',
 'umbers',
 '(',
 'a',
 ',',
 'Ġb',
 '):',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ"""',
 'Add',
 'Ġthe',
 'Ġtwo',
 'Ġnumbers',
 'Ġ`',
 'a',
 '`',
 'Ġand',
 'Ġ`',
 'b',
 '`',
 '."',
 '""',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġreturn',
 'Ġa',
 'Ġ+',
 'Ġb']

##  Step 14: Train the New Tokenizer

We use **`train_new_from_iterator()`** to train our new tokenizer on the Python code corpus.

- We pass our `training_corpus` generator.
- We specify `52000` as the target vocabulary size (matching GPT-2).

Because Hugging Face "fast" tokenizers are backed by Rust, this statistical process is incredibly fast, taking only a minute or two for over 1.6 GB of text!

In [12]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)

##  Step 15: Test the New Tokenizer on the Same Code

Let's tokenize the same example function using our **newly trained** Python tokenizer.

**Improvements:**
- It learned special tokens for Python constructs like `Ġ"""` for docstrings.
- It learned a single token `ĊĠĠĠ` representing a newline followed by an indentation.
- It correctly splits the function name on the underscore `_`.

In [13]:
tokens = tokenizer.tokenize(example)
tokens

['def',
 'Ġadd',
 '_',
 'numbers',
 '(',
 'a',
 ',',
 'Ġb',
 '):',
 'ĊĠĠĠ',
 'Ġ"""',
 'Add',
 'Ġthe',
 'Ġtwo',
 'Ġnumbers',
 'Ġ`',
 'a',
 '`',
 'Ġand',
 'Ġ`',
 'b',
 '`."""',
 'ĊĠĠĠ',
 'Ġreturn',
 'Ġa',
 'Ġ+',
 'Ġb']

##  Step 16: Compare Token Counts

Because our new tokenizer has a vocabulary optimized for Python code, it requires **fewer tokens** to represent the same snippet.

This makes your language model more efficient, as it can process more code within the same context window.

In [14]:
print(len(tokens))
print(len(old_tokenizer.tokenize(example)))

27
36


##  Step 17: Test on a PyTorch Class

Let's test it on a more complex Python snippet: a PyTorch `LinearLayer` class.

Notice how Python-specific keywords like `class`, `__init__`, `self`, `return`, and even double-indentations (`ĊĠĠĠĠĠĠĠ`) are efficiently tokenized into single tokens.

In [15]:
example2 = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """
tokenizer.tokenize(example2)

['class',
 'ĠLinear',
 'Layer',
 '():',
 'ĊĠĠĠ',
 'Ġdef',
 'Ġ__',
 'init',
 '__(',
 'self',
 ',',
 'Ġinput',
 '_',
 'size',
 ',',
 'Ġoutput',
 '_',
 'size',
 '):',
 'ĊĠĠĠĠĠĠĠ',
 'Ġself',
 '.',
 'weight',
 'Ġ=',
 'Ġtorch',
 '.',
 'randn',
 '(',
 'input',
 '_',
 'size',
 ',',
 'Ġoutput',
 '_',
 'size',
 ')',
 'ĊĠĠĠĠĠĠĠ',
 'Ġself',
 '.',
 'bias',
 'Ġ=',
 'Ġtorch',
 '.',
 'zeros',
 '(',
 'output',
 '_',
 'size',
 ')',
 'ĊĊĠĠĠ',
 'Ġdef',
 'Ġ__',
 'call',
 '__(',
 'self',
 ',',
 'Ġx',
 '):',
 'ĊĠĠĠĠĠĠĠ',
 'Ġreturn',
 'Ġx',
 'Ġ@',
 'Ġself',
 '.',
 'weights',
 'Ġ+',
 'Ġself',
 '.',
 'bias',
 'ĊĠĠĠĠ']

##  Step 18: Save the Tokenizer Locally

We use **`save_pretrained()`** to save the newly trained tokenizer to a local directory.
This creates a folder containing the `tokenizer.json` and config files so you can load it later.

In [16]:
tokenizer.save_pretrained("code-search-net-tokenizer")

('code-search-net-tokenizer\\tokenizer_config.json',
 'code-search-net-tokenizer\\tokenizer.json')

##  Step 19: Push the Tokenizer to the Hub

To share the tokenizer with the community (or use it easily across different machines), we push it to the Hugging Face Hub using **`push_to_hub()`**.

> *Note: You must have run `notebook_login()` successfully in Step 3 for this to work.*

In [17]:
tokenizer.push_to_hub("code-search-net-tokenizer")

CommitInfo(commit_url='https://huggingface.co/SAK-07/code-search-net-tokenizer/commit/5de8a5eb1662eaa7a37722d57d4fc03f5d5ada3f', commit_message='Upload tokenizer', commit_description='', oid='5de8a5eb1662eaa7a37722d57d4fc03f5d5ada3f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SAK-07/code-search-net-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='SAK-07/code-search-net-tokenizer'), pr_revision=None, pr_num=None)

##  Step 20: Load the Tokenizer from the Hub

Finally, you can load your custom tokenizer from anywhere using `AutoTokenizer.from_pretrained()`, pointing to your namespace and repository name.

You're now ready to train a language model on Python code from scratch!

In [18]:
# Replace "huggingface-course" below with your actual namespace to use your own tokenizer
tokenizer = AutoTokenizer.from_pretrained("SAK-07/code-search-net-tokenizer")

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sujat\.cache\huggingface\hub\models--SAK-07--code-search-net-tokenizer. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]